# Phase 4b — fix the tests, then fix the retrieval

Three jobs:

1. **Diagnose** — confirm the model truncates at 64 tokens while tafsir
   passages average 7,731 characters.
2. **Redo two broken tests** from the last notebook (wrong statistic,
   confounded ablation).
3. **Try the fix** — chunk the tafsir into overlapping windows, re-index,
   re-measure.

Runtime → T4 GPU → Run all. Sections 1–5 are fast; section 6 takes ~30 min.


## 1. Setup

In [ ]:
import torch, os, shutil, subprocess, sys, json, time
print("CUDA:", torch.cuda.is_available())
from google.colab import drive; drive.mount('/content/drive')

ROOT="/content/drive/MyDrive"
P1=f"{ROOT}/Phase1_Project/MemberB_B4_B6_output"
P1FIX=f"{ROOT}/Phase1_Project/data_fix_output"
GV2=f"{ROOT}/Phase3_Project/guardrail_output_v2"
OUT=f"{ROOT}/Phase4_Project"; os.makedirs(f"{OUT}/data", exist_ok=True)

PROJECT="/content/QuranicRAG"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(f"{PROJECT}/src", exist_ok=True)
os.makedirs(f"{PROJECT}/quranNLP/shared/data", exist_ok=True)
os.chdir(PROJECT)
shutil.rmtree("/content/_repo", ignore_errors=True)
subprocess.run(["git","clone","--depth","1",
 "https://github.com/Laiba-Noor/quranic-rag-hallucination-free.git","/content/_repo"],check=True)
for f in os.listdir("/content/_repo/src"):
    if f.endswith(".py"): shutil.copy(f"/content/_repo/src/{f}", f"src/{f}")
for f in os.listdir(f"{GV2}/src"):
    if f.endswith(".py"): shutil.copy(f"{GV2}/src/{f}", f"src/{f}")
shutil.copy(f"{P1FIX}/shared_data/final_cross_reference_index.csv",
            "quranNLP/shared/data/final_cross_reference_index.csv")
!pip install -q sentence-transformers hnswlib scipy pandas
print("ready")

In [ ]:
t=time.time()
shutil.copytree(f"{P1}/b5_real_finetuned",     "m_v1", dirs_exist_ok=True)
shutil.copytree(f"{P1}/index",                 "i_v1", dirs_exist_ok=True)
shutil.copytree(f"{GV2}/b5_real_finetuned_v2", "m_v2", dirs_exist_ok=True)
shutil.copytree(f"{GV2}/index_v2",             "i_v2", dirs_exist_ok=True)
print(f"copied in {time.time()-t:.0f}s")

sys.path.insert(0,"src")
from sentence_transformers import SentenceTransformer
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI
def build(mdir, idir):
    m=SentenceTransformer(mdir)
    idx,ent=load_index(dim=m.get_sentence_embedding_dimension(), out_dir=idir)
    return RetrievalAPI(m,idx,ent), ent, m
api_v1, ent1, mod1 = build("m_v1","i_v1")
api_v2, ent2, mod2 = build("m_v2","i_v2")
print(len(ent1), len(ent2))

## 2. THE DIAGNOSIS — how much text does the model actually see?

In [ ]:
import json, statistics as st

for name in ["m_v1","m_v2"]:
    cfg=json.load(open(f"{name}/sentence_bert_config.json"))
    print(f"{name}: max_seq_length = {cfg.get('max_seq_length')}")
print(f"\nlive objects: v1={mod1.max_seq_length}  v2={mod2.max_seq_length}")

tafsir=[e["text"] for e in ent2 if e["source_type"]=="tafsir"]
lens=sorted(len(t) for t in tafsir)
print(f"\ntafsir passages: {len(tafsir)}")
print(f"  median {st.median(lens):,.0f} chars | mean {st.mean(lens):,.0f} | max {lens[-1]:,}")

CHARS_PER_TOK = 3.9   # Arabic, GATE-AraBert tokenizer, approximate
seen = mod2.max_seq_length * CHARS_PER_TOK
print(f"\nmodel window ~= {seen:,.0f} chars")
print(f"fraction of the MEDIAN passage the model sees: {100*min(1,seen/st.median(lens)):.1f}%")
print(f"passages longer than the window: "
      f"{sum(1 for l in lens if l>seen)}/{len(lens)} "
      f"({100*sum(1 for l in lens if l>seen)/len(lens):.0f}%)")

## 3. Ground truth + metrics

In [ ]:
import math, os, shutil, json
STASH=f"{OUT}/data/ayatec_records.json"
cands=[STASH, f"{ROOT}/Phase2_Project/Roma_output/data/ayatec_records.json",
       f"{P1FIX}/shared_data/ayatec_records.json", "/content/_repo/Data/ayatec_records.json"]
path=next((c for c in cands if os.path.exists(c)), None)
if path is None:
    from google.colab import files; up=files.upload(); path=list(up.keys())[0]
if os.path.abspath(path)!=os.path.abspath(STASH): shutil.copy(path, STASH)
aya=json.load(open(STASH,encoding="utf-8"))
GOLD=[(r["question"], set(r["verse_keys"])) for r in aya if r.get("question") and r.get("verse_keys")]
print(len(GOLD),"questions")

def ranked_verse_keys(api, query, depth=50, skip_entry=None):
    seen,out=set(),[]
    for r in api.retrieve(query, top_k=depth):
        if skip_entry and skip_entry(r): continue
        vk=r["verse_key"]
        if vk not in seen: seen.add(vk); out.append(vk)
    return out

def evaluate(api, gold_pairs, ks=(1,5,10,20), depth=50, skip_entry=None):
    from collections import defaultdict
    acc=defaultdict(list); per_q={"rr":[], "r10":[], "hit20":[]}
    for q,gold in gold_pairs:
        ranked=ranked_verse_keys(api,q,depth,skip_entry)
        rr=next((1.0/i for i,vk in enumerate(ranked,1) if vk in gold), 0.0)
        acc["MRR"].append(rr)
        dcg=sum(1/math.log2(i+1) for i,vk in enumerate(ranked[:10],1) if vk in gold)
        idcg=sum(1/math.log2(i+1) for i in range(1,min(len(gold),10)+1))
        acc["NDCG@10"].append(dcg/idcg if idcg else 0.0)
        for k in ks:
            h=sum(1 for vk in ranked[:k] if vk in gold)
            acc[f"Recall@{k}"].append(h/len(gold)); acc[f"HitRate@{k}"].append(1.0 if h else 0.0)
        per_q["rr"].append(rr)
        per_q["r10"].append(sum(1 for vk in ranked[:10] if vk in gold)/len(gold))
        per_q["hit20"].append(1.0 if any(vk in gold for vk in ranked[:20]) else 0.0)
    return {m: sum(v)/len(v) for m,v in acc.items()}, per_q

ceil10 = sum(min(10,len(g))/len(g) for _,g in GOLD)/len(GOLD)
print(f"max achievable Recall@10 = {ceil10:.4f} (gold sets are large)")

## 4. FIX 1 — significance on the right statistic

Last time I ran Wilcoxon on reciprocal rank, which only cares where the
*first* correct verse lands. The claim is about the top 10/20, so test that.


In [ ]:
from scipy.stats import wilcoxon
import statistics as st

res1,pq1 = evaluate(api_v1, GOLD)
res2,pq2 = evaluate(api_v2, GOLD)

print(f"{'metric':<12}{'v1':>9}{'v2':>9}{'delta':>10}")
print("-"*40)
for m in ["Recall@1","Recall@5","Recall@10","Recall@20","HitRate@1",
          "HitRate@5","HitRate@10","HitRate@20","MRR","NDCG@10"]:
    print(f"{m:<12}{res1[m]:>9.4f}{res2[m]:>9.4f}{res2[m]-res1[m]:>+10.4f}")

print("\n--- paired tests, v2 vs v1 ---")
for label in ["rr","r10","hit20"]:
    a,b = pq1[label], pq2[label]
    d=[y-x for x,y in zip(a,b)]
    if not any(d): print(f"{label:<7} identical"); continue
    stat,p = wilcoxon(a,b, zero_method="wilcox")
    sd=st.pstdev(d) or 1e-9
    print(f"{label:<7} mean delta={st.mean(d):+.4f}  p={p:.4g}  d={st.mean(d)/sd:+.3f}  "
          f"{'SIGNIFICANT' if p<0.05 else 'not significant'}")

## 5. FIX 2 — ablation at entry level

Last time I filtered by `verse_key`, which also deleted the Qur'anic verse for
those 630 keys. Filter the header *entry* instead, keep the verse.


In [ ]:
is_header = lambda r: (r["source_type"]=="tafsir"
    and r["text"].lstrip().startswith(("تَفْسِيرُ سُورَةِ","تفسير سورة")))

print(f"{'system':<24}{'Recall@10':>11}{'HitRate@20':>12}{'MRR':>9}{'NDCG@10':>10}")
print("-"*66)
for name, api in [("v1",api_v1), ("v2",api_v2)]:
    for tag, skip in [("as-is",None), ("no header entries",is_header)]:
        r,_ = evaluate(api, GOLD, skip_entry=skip)
        print(f"{name+' ('+tag+')':<24}{r['Recall@10']:>11.4f}"
              f"{r['HitRate@20']:>12.4f}{r['MRR']:>9.4f}{r['NDCG@10']:>10.4f}")

---
## 6. THE EXPERIMENT — chunk the tafsir and re-index

If the diagnosis is right, the fix is to stop feeding 7,000-character
passages into a 64-token window. Split each passage into overlapping windows,
raise the window to 512 tokens, and index the chunks.

Unique passages are encoded once and shared, so this stays affordable.
**~30 minutes.**


In [ ]:
import csv, sys, numpy as np, hnswlib, pickle, time
csv.field_size_limit(sys.maxsize)

WIN, STRIDE, MAXSEQ = 1000, 800, 512

def chunks(text):
    if len(text) <= WIN: return [text]
    return [text[i:i+WIN] for i in range(0, len(text), STRIDE) if text[i:i+WIN].strip()]

rows=list(csv.DictReader(open("quranNLP/shared/data/final_cross_reference_index.csv",
                              encoding="utf-8")))
uniq={}                      # passage -> list of chunk strings
for r in rows:
    p=(r.get("tafsir_passage") or "").strip()
    if p and str(r.get("has_direct_tafsir"))=="True" and p not in uniq:
        uniq[p]=chunks(p)

all_chunks=[]; chunk_ix={}
for p,cs in uniq.items():
    ids=[]
    for c in cs:
        if c not in chunk_ix:
            chunk_ix[c]=len(all_chunks); all_chunks.append(c)
        ids.append(chunk_ix[c])
    uniq[p]=ids

print(f"unique passages: {len(uniq)}")
print(f"unique chunks to encode: {len(all_chunks)}")
print(f"avg chunks per passage: {sum(len(v) for v in uniq.values())/len(uniq):.1f}")

In [ ]:
model = SentenceTransformer("m_v2")
model.max_seq_length = MAXSEQ
print("max_seq_length now:", model.max_seq_length)

t=time.time()
chunk_emb = model.encode(all_chunks, convert_to_numpy=True, normalize_embeddings=True,
                         show_progress_bar=True, batch_size=32)
print(f"chunks encoded in {(time.time()-t)/60:.1f} min")

verses=[]; 
for r in rows:
    v=(r.get("clean_verse") or "").strip()
    if v: verses.append((r["verse_key"], v))
verse_emb = model.encode([v for _,v in verses], convert_to_numpy=True,
                         normalize_embeddings=True, show_progress_bar=True, batch_size=64)
print("verses encoded:", verse_emb.shape)

In [ ]:
entries=[]; vecs=[]
for (vk, vtext), e in zip(verses, verse_emb):
    entries.append({"text":vtext,"verse_key":vk,"source_type":"verse"}); vecs.append(e)

for r in rows:
    p=(r.get("tafsir_passage") or "").strip()
    if not p or str(r.get("has_direct_tafsir"))!="True": continue
    for cid in uniq[p]:
        entries.append({"text":all_chunks[cid],"verse_key":r["verse_key"],"source_type":"tafsir"})
        vecs.append(chunk_emb[cid])

vecs=np.vstack(vecs)
print(f"total entries: {len(entries)}  (was 12,472)")

idx=hnswlib.Index(space="cosine", dim=vecs.shape[1])
idx.init_index(max_elements=len(entries), ef_construction=200, M=16)
idx.add_items(vecs, ids=np.arange(len(entries)))
idx.set_ef(64)
api_chunk = RetrievalAPI(model, idx, entries)
print("chunked index built")

## 7. Did chunking help?

In [ ]:
res3, pq3 = evaluate(api_chunk, GOLD, depth=100)

print(f"{'metric':<12}{'v2 orig':>10}{'v2 chunk':>11}{'delta':>10}")
print("-"*43)
for m in ["Recall@1","Recall@5","Recall@10","Recall@20","HitRate@1",
          "HitRate@5","HitRate@10","HitRate@20","MRR","NDCG@10"]:
    print(f"{m:<12}{res2[m]:>10.4f}{res3[m]:>11.4f}{res3[m]-res2[m]:>+10.4f}")

print("\n--- paired tests, chunked vs v2 ---")
for label in ["r10","hit20"]:
    a,b=pq2[label],pq3[label]
    d=[y-x for x,y in zip(a,b)]
    if any(d):
        stat,p=wilcoxon(a,b,zero_method="wilcox"); sd=st.pstdev(d) or 1e-9
        print(f"{label:<7} mean delta={st.mean(d):+.4f}  p={p:.4g}  d={st.mean(d)/sd:+.3f}  "
              f"{'SIGNIFICANT' if p<0.05 else 'not significant'}")
print(f"\nRecall@10 as % of achievable ({ceil10:.3f}): "
      f"v2 {100*res2['Recall@10']/ceil10:.1f}%  ->  chunked {100*res3['Recall@10']/ceil10:.1f}%")

## 8. Save everything

In [ ]:
import pickle, os, json
os.makedirs(f"{OUT}/index_chunked", exist_ok=True)
idx.save_index(f"{OUT}/index_chunked/verses.hnsw")
pickle.dump(entries, open(f"{OUT}/index_chunked/entries.pkl","wb"))
json.dump({"n_questions":len(GOLD),"ceiling_recall10":ceil10,
           "v1":res1,"v2":res2,"v2_chunked":res3,
           "config":{"window":WIN,"stride":STRIDE,"max_seq_length":MAXSEQ,
                     "n_entries":len(entries)}},
          open(f"{OUT}/phase4_chunking_experiment.json","w"), indent=2, ensure_ascii=False)
print("saved to", OUT)